# Validation Test Simulation Setup - Rising Bubble Benchmark Testcase 1

This notebook and the corresponding evaluation notebook (RisingBubble_Testcase1_Evaluation.ipynb) are part of published results (cf. 7.3.1) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853). We compare our results against the rising bubble benchmark test case established by Hysing et al ( https://doi.org/10.1002/fld.1934)

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [3]:
static string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
//var myBatch = ExecutionQueues[1];
static var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal
SingleNode,True


In [5]:
wmg.Init("XNSEpaper_RisingBubble", myBatch);

Project name is set to 'XNSEpaper_RisingBubble'.
trying to run from backup database...
   Bkup Database dirs: \\dc3\userspace\smuda\hpccluster\bkup-2025Oct14_000000.XNSEpaper_RisingBubble;
Selecting newest available backup: \\dc3\userspace\smuda\hpccluster\bkup-2025Oct14_000000.XNSEpaper_RisingBubble
Opening existing database '\\dc3\userspace\smuda\hpccluster\bkup-2025Oct14_000000.XNSEpaper_RisingBubble'.


In [6]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [7]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [8]:
static var sessions = wmg.Sessions;
sessions

#0: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80*	3/23/2020 10:05:28 PM	931391e0...
#1: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20	3/23/2020 10:05:20 PM	d3c1cfe5...
#2: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40	3/23/2020 10:05:22 PM	44504bf6...
#3: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart1*	4/17/2020 2:43:50 PM	7a7385d1...
#4: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart2*	5/31/2020 8:10:24 PM	97a64a0d...
#5: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart3*	6/1/2020 4:00:14 PM	42b883f6...
#6: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart4*	6/3/2020 10:12:02 PM	22177ad6...
#7: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart5	6/5/2020 9:21:58 PM	6e3cf7d8...
#8: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20*	5/8/2020 6:45:50 PM	8351c0ba...
#9: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20_restart1	5/9

In [9]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("RisingBubble") && s.CreationTime.Date == new DateTime(2026, 2, 4) && !s.SuccessfulTermination) {
//         s.Delete(true);
//         Console.WriteLine($"Deleted session {s.Name} from {s.CreationTime.Date}");
//     }
// }

In [10]:
// static var sessions = wmg.Sessions;
// sessions

## Grid Creation

The initial setup is depicted in Figure 7, where a circular bubble (radius $r = 0.25$) is positioned at $(0.5, 0.5)$ in the domain $\Omega = [0, 1] × [0, 2]$.
The lower and upper boundary are imposed by the wall boundary condition. The boundaries at the sides describe a free-slip boundary condition given by
>$$
\vec{u} \cdot \vec{n}_{\partial \Omega} = 0, \quad \vec{\tau}_{\partial \Omega} \cdot (\nabla \vec{u} + \nabla \vec{u}^T) \vec{n}_{\partial \Omega} = 0
$$<
where $\vec{\tau}_{\partial \Omega}$ denotes the tangent vector on the boundary $\partial \Omega$.

In [11]:
bool fullDomain = true;

In [12]:
int[] Resolutions = new int[] { 10, 20, 40, 60, 80 };
//int[] Resolutions = new int[] { 20 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"RisingBubble2D_{Res}x{2*Res}";
    GridName = fullDomain ? GridName + "_fullDomain" : "_halfDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = fullDomain ? GenericBlas.Linspace(0, 1.0, Res + 1) : GenericBlas.Linspace(0.5, 1.0, Res + 1);
        double[] yNodes = GenericBlas.Linspace(0, 2.0, Res*2 + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;

            if (fullDomain) {
                if(X.x.Abs() <= 1e-8)
                ret = IncompressibleBcType.FreeSlip.ToString();
            } else {
                if((X.x - 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            }
            if((X.x - 1.0).Abs() <= 1e-8)
                ret = IncompressibleBcType.FreeSlip.ToString();
            if((X.y).Abs() <= 1e-8)         // lower wall
                ret = IncompressibleBcType.Wall.ToString();
            if((X.y - 2.0).Abs() <= 1e-8)   // upper wall
                ret = IncompressibleBcType.Wall.ToString();

            return ret;
        });    
        
        // grd.AddPredefinedPartitioning("FourProcSplit_fullDomain", delegate (double[] X) {
        //     int rank;
        //     double x = X[0]; double y = X[1];
        //     if (x < 0.5 && y < 0.5)
        //         rank = 0;
        //     else if (x >= 0.5 && y < 0.5)
        //         rank = 1;
        //     else if (x < 0.5 && y >= 0.5)
        //         rank = 2;
        //     else 
        //         rank = 3;

        //     return rank;
        // });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Grid Edge Tags changed.
Running in bacvkup mode, and no equivalent grid is found.
Grid Edge Tags changed.
Running in bacvkup mode, and no equivalent grid is found.
Grid Edge Tags changed.
Running in bacvkup mode, and no equivalent grid is found.
Grid Edge Tags changed.
Running in bacvkup mode, and no equivalent grid is found.
Grid Edge Tags changed.
Running in bacvkup mode, and no equivalent grid is found.


## Initial Values

In [13]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.5).Pow2() + (X[1] - 0.5).Pow2()) - dropRadius; " + 
    "}");

The driving force for the rise of the bubble ($\rho_{\mathfrak{A}} < \rho_{\mathfrak{B}}$) is the gravity force $\vec{g} = −g\vec{e}_y$
oriented in negative $y$-direction.

In [14]:
var GravityValue = new Formula(
    "grav",
    false,
    "using ilPSP.Utils; " + 
    "double grav(double[] X) { return -0.98; }");

## Physical Settings

Two test cases are defined with different physical properties resulting in an ellipsoidal terminal shape for the first test case and a dimpled cap with filaments for the second test case. The corresponding physical
parameters are given in Table 4. For both test cases the simulation is performed for three time units.

In [15]:
public class PhysicalSettings {

    public double rhoA;      // bubble
    public double rhoB;   // surrounding fluid
    public double muA;
    public double muB;
    public double sigma;

    public PhysicalSettings(string testcase) {
        switch(testcase) {
            case "tc1": { // ellipsoidal terminal shape
                rhoA = 100.0;
                rhoB = 1000.0;
                muA = 1.0;
                muB = 10.0;
                sigma = 24.5;

                break;
            }
            case "tc2": { // dimpled cap with filaments 
                rhoA = 1.0;
                rhoB = 1000.0;
                muA = 0.1;
                muB = 10.0;
                sigma = 1.96;
                
                break;
            }
        }
    }
}

## Control settings

In [16]:
public class StudySettings {

    public string CaseName;
    public PhysicalSettings PhysicalParams;

    public int degree;
    public int grdIdx;

    public double dt;
    public int savePeriod;
    public int logPeriod;

    public StudySettings(string testcase, int k, int gridIndex, double timeStep, int savePer, int logPer) {
        CaseName = testcase;
        PhysicalParams = new PhysicalSettings(testcase);

        degree = k;
        grdIdx = gridIndex;     

        dt = timeStep;
        savePeriod = savePer;   
        logPeriod = logPer;
    }

}

In [17]:
static double t_end = 3.0;

In [18]:
//int[] Resolutions = new int[] { 10, 20, 40, 60, 80 };

In [19]:
List<StudySettings> allStudies = new List<StudySettings>();
// testcase1 studies
allStudies.Add(new StudySettings("tc1", 2, 1, 0.005, 10, 1));    //mesh20
allStudies.Add(new StudySettings("tc1", 2, 2, 0.0015, 20, 2));    //mesh40
allStudies.Add(new StudySettings("tc1", 2, 3, 0.001, 30, 3));    //mesh60
//allStudies.Add(new StudySettings("tc1", 2, 4, 0.0006, 50, 5));    //mesh80

allStudies.Add(new StudySettings("tc1", 3, 0, 0.006, 10, 1));    //mesh10
allStudies.Add(new StudySettings("tc1", 3, 1, 0.003, 10, 1));    //mesh20
allStudies.Add(new StudySettings("tc1", 3, 2, 0.001, 30, 3));    //mesh40
//allStudies.Add(new StudySettings("tc1", 3, 3, 0.0006, 50, 5));    //mesh60

In [20]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [21]:
foreach(StudySettings study in allStudies) {

    XNSE_Control C = new XNSE_Control();

    C.SetDGdegree(study.degree);
    
    C.FieldOptions.Add(VariableNames.GravityY + "#A", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    C.FieldOptions.Add(VariableNames.GravityY + "#B", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });

    // physical parameters
    C.PhysicalParameters.rho_A = study.PhysicalParams.rhoA;
    C.PhysicalParameters.mu_A = study.PhysicalParams.muA;
    
    C.PhysicalParameters.rho_B = study.PhysicalParams.rhoB;
    C.PhysicalParameters.mu_B = study.PhysicalParams.muB;
    
    C.PhysicalParameters.Sigma = study.PhysicalParams.sigma;
    
    C.PhysicalParameters.IncludeConvection = true;
  

    // set grid
    C.SetGrid(Grids[study.grdIdx]);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   
         
    C.AddInitialValue("GravityY#A", GravityValue);
    C.AddInitialValue("GravityY#B", GravityValue);
        
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;
 
    if (study.CaseName == "tc1" && study.degree == 3 && Resolutions[study.grdIdx] == 10) {
        C.FastMarchingReInitPeriod = 50;
    }

    C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;

    C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); 
    // C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    // C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.MaxSolverIterations = 50;


    if (study.CaseName == "tc2") {
        C.AdaptiveMeshRefinement = true;
        C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = 1 });
        C.AMR_startUpSweeps = 1;
    }


    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    C.dtFixed = study.dt;
    C.Endtime = t_end;
    C.NoOfTimesteps = (int) (t_end / study.dt);

    C.saveperiod = study.savePeriod;


    C.PostprocessingModules.Add(new RisingBubble2DBenchmarkQuantities(){ SolverStage = 2, LogPeriod = study.logPeriod });
   
    C.SessionName = $"RisingBubble_{study.CaseName}_ConvStudy_k{study.degree}_mesh{Resolutions[study.grdIdx]}";
    if (study.CaseName == "tc1" && study.degree == 3 && Resolutions[study.grdIdx] == 10) {  
        C.SessionName = $"RisingBubble_{study.CaseName}_ConvStudy_k{study.degree}_mesh{Resolutions[study.grdIdx]}_ReInit50";
    }
    
    Controls.Add(C);
}

In [22]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
}

RisingBubble_tc1_ConvStudy_k2_mesh20
RisingBubble_tc1_ConvStudy_k2_mesh40
RisingBubble_tc1_ConvStudy_k2_mesh60
RisingBubble_tc1_ConvStudy_k3_mesh10_ReInit50
RisingBubble_tc1_ConvStudy_k3_mesh20
RisingBubble_tc1_ConvStudy_k3_mesh40


In [23]:
int NoCtrls = Controls.Count();
NoCtrls

6

## Launch/Restart Jobs

In [24]:
public static Job GetLatestJob(Job job2rest) {

    string baseName = job2rest.LatestSession.Name;

    // find latest session with same base name (in case of multiple restarts)
    var studySess = sessions.Where(sess => sess.Name.Contains(baseName));
    int studyCount = studySess.Count();
    if (studyCount == 1)
        return job2rest;
        
    studySess = studySess.OrderBy(sess => sess.CreationTime);
    var LatestSession = studySess.Last();

    var latestCtrl = LatestSession.GetControl();
    var latestJob = latestCtrl.CreateJob();

    return latestJob;
}

In [25]:
public static ITimestepInfo GetLatestTimestepWithDesiredRestartSteps(ISessionInfo sess, int desiredRestartSteps) {
    var timesteps = sess.Timesteps.WithoutSubSteps().TakeLast(10);
    for (int i = timesteps.Count() - 1; i >= 0; i--) {
        var ts = timesteps.Pick(i);
        int tsNum = ts.TimeStepNumber.MajorNumber;
        Console.WriteLine($"Timestep: {tsNum}");
        for (int j = 1; j <= desiredRestartSteps; j++) {
            if (i >= j) {
                var rts = timesteps.Pick(i - j);
                int rtsNum = tsNum - j;
                if (rts.TimeStepNumber.MajorNumber == rtsNum) {
                    if (j == desiredRestartSteps) {
                        Console.WriteLine($"Found timestep {tsNum} with {desiredRestartSteps} restart steps.");
                        return ts;
                    }
                } else {
                    break;
                }
            } else {
                break;
            }
        }
    }
    throw new Exception("No timestep with desired restart steps found.");
}

In [26]:
public static Job CreateRestartJob(Job job2rest) {

    string baseName = job2rest.LatestSession.Name;

    // find latest session with same base name (in case of multiple restarts)
    var studySess = sessions.Where(sess => sess.Name.Contains(baseName));
    int studyCount = studySess.Count();
    studySess = studySess.OrderBy(sess => sess.CreationTime);
    var LatestSession = studySess.Last();
    // // check if latest session terminated successfully
    // if (LatestSession.SuccessfulTermination) {
    //     Console.WriteLine($"Latest session {LatestSession.Name} already terminated successfully; no restart job created.");
    //     return LatestSession.GetControl().CreateJob();
    // }
    // Console.WriteLine($"Creating restart job for session {LatestSession.Name} ...");

    var rCtrl = LatestSession.GetControl().CloneAs();
    // var rCtrl = job2rest.LatestSession.GetControl().CloneAs();

    // Guid rst_ID = LatestSession.ID;
    // TimestepNumber rst_ts = LatestSession.Timesteps.Last().TimeStepNumber;

    Guid rst_ID = LatestSession.ID;
    //var LastTimestep = LatestSession.Timesteps.Last();
    var LastTimestep = GetLatestTimestepWithDesiredRestartSteps(LatestSession, 2);  // for BDF3
    TimestepNumber rst_ts = LastTimestep.TimeStepNumber;

    rCtrl.GridGuid = LastTimestep.GridID;
    rCtrl.m_Grid = LastTimestep.Grid as IGrid;

    rCtrl.InitialValues.Clear();
    rCtrl.InitialValues_Evaluators.Clear();

    rCtrl.Endtime = t_end;
    rCtrl.RestartInfo = Tuple.Create(rst_ID, rst_ts);

    rCtrl.SessionName = baseName + "_Restart" + studyCount;

    // changes
    // =======
    ((XNSE_Control)rCtrl).LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); 
    // =======

    var rJob = rCtrl.CreateJob();

    // set resources
    rJob.NumberOfMPIProcs = job2rest.NumberOfMPIProcs;

    int numThreads = job2rest.NumberOfThreads;
    rJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = rJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    rJob.MySetCommandLineArguments(args.Values.ToArray());

    rJob.RetryCount = job2rest.RetryCount;

    Console.WriteLine($"Restart job for session {rCtrl.SessionName} created.");

    return rJob;
}

In [27]:
List<Job> jobs = new List<Job>();

In [28]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
    
    if (oneJob.Status == JobStatus.FailedOrCanceled) {
        var latestJob = GetLatestJob(oneJob);
        // in case latest job is the same as oneJob (i.e. no previous sessions exist)
        if (latestJob.Name == oneJob.Name) {
            Console.WriteLine($"Job for latest session {latestJob.Name} failed; create restart job ...");
            var restartJob = CreateRestartJob(oneJob);
            restartJob.Activate(myBatch);
            jobs.Add(restartJob);
        } else {
            latestJob.Activate(myBatch);

            if (latestJob.Status == JobStatus.InProgress) {
                Console.WriteLine($"Latest session {latestJob.Name} is still running; no restart job created.");
                jobs.Add(latestJob);
            }
            if (latestJob.Status == JobStatus.FinishedSuccessful) {
                Console.WriteLine($"Latest session {latestJob.Name} already finished successfully; no restart job created.");
                jobs.Add(latestJob);
            }
            if (latestJob.Status == JobStatus.FailedOrCanceled) {
                Console.WriteLine($"Job for latest session {latestJob.Name} failed; create restart job ...");
                var restartJob = CreateRestartJob(oneJob);
                restartJob.Activate(myBatch);
                jobs.Add(restartJob);
            }
        }
        //Console.WriteLine($"Job for session {ctrl.SessionName} failed. Trying to restart latest job ...");

        // var restartJob = CreateRestartJob(oneJob);
        // restartJob.Activate(myBatch);
        // jobs.Add(restartJob);

    } else {   
          
        //oneJob.Activate(myBatch);
        jobs.Add(oneJob);
    }
}

Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh20	3/23/2020 10:43:07 AM	6106f0e5..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh40	3/23/2020 10:43:12 AM	97e8e806..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-Wi

## Wait for Completion and Check Job Status

In [29]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                                                || job.Status == JobStatus.PendingInExecutionQueue
                                                                || job.Status == JobStatus.PreActivation).Count();
NoInProgress

0

In [30]:
int maxDays = 5;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

Error: (3,8): error CS0103: The name 'NoInProgress' does not exist in the current context
(4,5): error CS0103: The name 'wmg' does not exist in the current context
(5,5): error CS0103: The name 'NoInProgress' does not exist in the current context
(5,20): error CS0103: The name 'jobs' does not exist in the current context
(5,52): error CS0103: The name 'JobStatus' does not exist in the current context

In [31]:
int NoFailed = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

Error: (1,16): error CS0103: The name 'jobs' does not exist in the current context
(1,48): error CS0103: The name 'JobStatus' does not exist in the current context

In [32]:
int NoSuccess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

Error: (1,17): error CS0103: The name 'jobs' does not exist in the current context
(1,49): error CS0103: The name 'JobStatus' does not exist in the current context

In [33]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

Error: (2,5): error CS0103: The name 'NoFailed' does not exist in the current context
(3,26): error CS0103: The name 'Controls' does not exist in the current context
(5,27): error CS0103: The name 'JobStatus' does not exist in the current context
(6,29): error CS0103: The name 'sessions' does not exist in the current context
(14,21): error CS0103: The name 'NoFailed' does not exist in the current context
(15,21): error CS0103: The name 'NoSuccess' does not exist in the current context

In [34]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

Error: (1,1): error CS0103: The name 'NUnit' does not exist in the current context
(1,36): error CS0103: The name 'NoFailed' does not exist in the current context

In [35]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");

Error: (1,1): error CS0103: The name 'NUnit' does not exist in the current context
(1,33): error CS0103: The name 'NoCtrls' does not exist in the current context
(1,42): error CS0103: The name 'NoSuccess' does not exist in the current context